# Week 6 Assignment
## Spark Architecture and Efficient Data Processing

### Objective

The objective of this assignment is to understand Apache Spark architecture, Lazy Evaluation, DAG execution, DataFrame transformations, schema handling, optimized storage formats, and performance optimization techniques. The assignment also demonstrates building a complete Spark data processing pipeline using PySpark.

## Import Required Libraries

In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Create Spark Session

The SparkSession acts as the entry point for all Spark operations. It initializes the Driver process and allows interaction with Spark DataFrames.

In [24]:
spark = SparkSession.builder \
    .appName("Week 6 Assignment") \
    .getOrCreate()

print("Spark Version :", spark.version)

Spark Version : 4.2.0


## Load Dataset

The dataset is loaded from a CSV file with schema inference enabled. The first row is treated as the header.

In [25]:
import pandas as pd

# Read CSV using pandas (handles Latin-1 encoding correctly)
pdf = pd.read_csv(
    "data/source.csv",
    encoding="latin1"
)

# Convert pandas DataFrame to Spark DataFrame
df = spark.createDataFrame(pdf)

# Display first 5 rows
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Display Schema

In [26]:
df.printSchema()

root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



## Dataset Statistics

In [27]:
print("Number of Records :", df.count())
print("Number of Columns :", len(df.columns))

Number of Records : 9994
Number of Columns : 21


## Data Cleaning and Transformation

In this section, the dataset is modified by renaming columns, changing data types, and creating a new calculated column. These operations are common preprocessing steps before performing data analysis.

In [28]:
# Rename columns for easier querying

df = df.withColumnRenamed("Product ID", "product_id") \
       .withColumnRenamed("Sales", "price") \
       .withColumnRenamed("Category", "category")

print("Columns renamed successfully.")
df.printSchema()

Columns renamed successfully.
root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



### Output

The columns **Product ID**, **Sales**, and **Category** have been renamed to **product_id**, **price**, and **category** respectively.

In [29]:
from pyspark.sql.functions import col

df = df.withColumn(
    "price",
    col("price").cast("double")
)

df.printSchema()

root
 |-- Row ID: long (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: long (nullable = true)
 |-- Region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



### Output

The **price** column is now stored as a Double data type, making it suitable for mathematical calculations.

In [30]:
# Add 18% GST

df = df.withColumn(
    "final_price",
    col("price") * 1.18
)

df.select(
    "product_id",
    "price",
    "final_price"
).show(5)

+---------------+--------+------------------+
|     product_id|   price|       final_price|
+---------------+--------+------------------+
|FUR-BO-10001798|  261.96|309.11279999999994|
|FUR-CH-10000454|  731.94|          863.6892|
|OFF-LA-10000240|   14.62|           17.2516|
|FUR-TA-10000577|957.5775|        1129.94145|
|OFF-ST-10000760|  22.368|26.394239999999996|
+---------------+--------+------------------+
only showing top 5 rows


In [31]:
df.select(
    "product_id",
    "price"
).show(10)

+---------------+--------+
|     product_id|   price|
+---------------+--------+
|FUR-BO-10001798|  261.96|
|FUR-CH-10000454|  731.94|
|OFF-LA-10000240|   14.62|
|FUR-TA-10000577|957.5775|
|OFF-ST-10000760|  22.368|
|FUR-FU-10001487|   48.86|
|OFF-AR-10002833|    7.28|
|TEC-PH-10002275| 907.152|
|OFF-BI-10003910|  18.504|
|OFF-AP-10002892|   114.9|
+---------------+--------+
only showing top 10 rows


In [32]:
electronics_df = df.filter(
    col("category") == "Technology"
)

electronics_df.select(
    "product_id",
    "price"
).show(10)

+---------------+--------+
|     product_id|   price|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
|TEC-PH-10000486| 371.168|
|TEC-PH-10004093| 147.168|
|TEC-AC-10000171|   45.98|
|TEC-AC-10002167|    45.0|
|TEC-PH-10003988|    21.8|
+---------------+--------+
only showing top 10 rows


In [33]:
print("Total Records :", df.count())

Total Records : 9994


In [34]:
from pyspark.sql.functions import isnan, when, count

df.select([
    count(
        when(
            col(c).isNull(),
            c
        )
    ).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-----------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|product_id|category|Sub-Category|Product Name|price|Quantity|Discount|Profit|final_price|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-----------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|          0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+-------

## Save Processed Dataset as CSV

The processed dataset is written to CSV format with headers enabled. Spark automatically creates the output directory if it does not already exist.

In [35]:
import os

print("HADOOP_HOME =", os.environ.get("HADOOP_HOME"))

HADOOP_HOME = None


In [36]:
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("data/output/csv")

NameError: name 'filtered_df' is not defined

In [ ]:
filtered_df.explain()

== Physical Plan ==
LocalTableScan [Row ID#431L, Order ID#432, Order Date#433, Ship Date#434, Ship Mode#435, Customer ID#436, Customer Name#437, Segment#438, Country#439, City#440, State#441, Postal Code#442L, Region#443, product_id#541, category#543, Sub-Category#446, Product Name#447, price#544, Quantity#449L, Discount#450, Profit#451, final_price#545]




In [ ]:
lazy_df = df.filter(col("price") > 1000)
print("Transformation Created")
lazy_df.show()

Transformation Created
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|       State|Postal Code| Region|     product_id|       category|Sub-Category|        Product Name|   price|Quantity|Discount|    Profit|       final_price|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+------------------+
|    11|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|  Brosina Hoffman|   Consumer|United States|

## Spark Execution Plan

Apache Spark uses the Catalyst Optimizer to generate an optimized execution plan. The `explain()` method displays the logical and physical execution plan of the DataFrame.

In [ ]:
filtered_df.explain(True)

== Parsed Logical Plan ==
'Filter '`>`('price, 500)
+- Project [Row ID#431L, Order ID#432, Order Date#433, Ship Date#434, Ship Mode#435, Customer ID#436, Customer Name#437, Segment#438, Country#439, City#440, State#441, Postal Code#442L, Region#443, product_id#541, category#543, Sub-Category#446, Product Name#447, price#544, Quantity#449L, Discount#450, Profit#451, (price#544 * 1.18) AS final_price#545]
   +- Project [Row ID#431L, Order ID#432, Order Date#433, Ship Date#434, Ship Mode#435, Customer ID#436, Customer Name#437, Segment#438, Country#439, City#440, State#441, Postal Code#442L, Region#443, product_id#541, category#543, Sub-Category#446, Product Name#447, cast(price#542 as double) AS price#544, Quantity#449L, Discount#450, Profit#451]
      +- Project [Row ID#431L, Order ID#432, Order Date#433, Ship Date#434, Ship Mode#435, Customer ID#436, Customer Name#437, Segment#438, Country#439, City#440, State#441, Postal Code#442L, Region#443, product_id#541, Category#445 AS category#

## Lazy Evaluation

Spark does not execute transformations immediately. It builds a Directed Acyclic Graph (DAG) and executes it only when an action is performed.

In [ ]:
lazy_df = df.filter(col("price") > 1000)

print("Transformation created successfully.")
print("No computation has occurred yet.")

Transformation created successfully.
No computation has occurred yet.


In [ ]:
lazy_df.show(5)


+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-------------+------------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+----------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|         City|       State|Postal Code| Region|     product_id|  category|Sub-Category|        Product Name|   price|Quantity|Discount|    Profit|       final_price|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-------------+------------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+----------+------------------+
|    11|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|Brosina Hoffman| Consumer|United States|  Los Angeles|  California|      90032|   West|FUR-TA-

## Transformations and Actions

Transformations create a new DataFrame but are lazily evaluated. Actions trigger the execution of all pending transformations.

In [ ]:
selected_df = df.select("product_id", "price")
filtered_df = df.filter(col("price") > 100)

selected_df.show(5)

print("Total Records:", filtered_df.count())

+---------------+--------+
|     product_id|   price|
+---------------+--------+
|FUR-BO-10001798|  261.96|
|FUR-CH-10000454|  731.94|
|OFF-LA-10000240|   14.62|
|FUR-TA-10000577|957.5775|
|OFF-ST-10000760|  22.368|
+---------------+--------+
only showing top 5 rows
Total Records: 3765


## Difference Between show() and collect()

- `show()` displays a few rows of the DataFrame.
- `collect()` retrieves all rows to the driver and should only be used for small datasets.

In [ ]:
print("Using show()")

df.show(5)



Using show()
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     product_id|       category|Sub-Category|        Product Name|   price|Quantity|Discount|  Profit|       final_price|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42

## Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

Apache Spark follows a distributed architecture consisting of three main components:

- **Driver:** The Driver is the main process of a Spark application. It creates the SparkSession, converts user code into execution plans, schedules tasks, and collects the final results.
- **Cluster Manager:** The Cluster Manager allocates computing resources to the Spark application. Spark supports Standalone, YARN, Mesos, and Kubernetes as cluster managers.
- **Executor:** Executors run on worker nodes and execute the tasks assigned by the Driver. They also store cached data in memory and return results back to the Driver.

Together, these components enable scalable and fault-tolerant distributed processing.

## Q2. How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

Spark does not execute transformations immediately. Instead, it records all transformations and builds a Directed Acyclic Graph (DAG). Execution begins only when an action such as `show()`, `count()`, or `collect()` is called.

Benefits:
- Reduces unnecessary computations.
- Optimizes execution using the Catalyst Optimizer.
- Minimizes disk I/O.
- Improves overall performance.

In [ ]:
# Q.3 Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

df = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)

## Q4. Difference between CSV and Parquet

| CSV | Parquet |
|------|----------|
| Row-based storage | Columnar storage |
| Larger file size | Better compression |
| Slower query performance | Faster analytical queries |
| No schema information | Stores schema |
| Human readable | Binary format |

Parquet reads only the required columns, making it much faster and more efficient for big data processing.

In [ ]:
# Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Technology'.

from pyspark.sql.functions import col

df.select(
    "Product ID",
    "Sales"
).filter(
    col("Category") == "Technology"
).show()

+---------------+--------+
|     Product ID|   Sales|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
|TEC-PH-10000486| 371.168|
|TEC-PH-10004093| 147.168|
|TEC-AC-10000171|   45.98|
|TEC-AC-10002167|      45|
|TEC-PH-10003988|    21.8|
|TEC-PH-10002447| 1029.95|
|TEC-AC-10002167|      30|
|TEC-AC-10004633|   13.98|
|TEC-PH-10002726| 167.968|
|TEC-AC-10001998|   19.99|
|TEC-PH-10004093|  73.584|
|TEC-AC-10001767|  95.976|
|TEC-AC-10001552| 238.896|
|TEC-AC-10003499|  74.112|
|TEC-PH-10002844|  27.992|
+---------------+--------+
only showing top 20 rows


In [ ]:
# Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double. 

from pyspark.sql.functions import col

df = df.withColumnRenamed(
    "Product ID",
    "product_id"
)

df = df.withColumnRenamed(
    "Sales",
    "price"
)

df = df.withColumn(
    "price",
    col("price").cast("double")
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     product_id|       Category|Sub-Category|        Product Name|   price|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

## Q7. How does Spark use the DAG for fault tolerance?

Spark maintains a Lineage Graph (DAG) that records every transformation applied to a dataset. If a worker node fails, Spark does not reload the entire dataset. Instead, it recomputes only the lost partitions using the lineage information. This provides efficient fault tolerance without replicating all intermediate data.

### Q8. Write a query to filter a DataFrame `df_orders` for rows where the status is 'Completed' AND the amount is greater than 1000.

```python
from pyspark.sql.functions import col

df_orders.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
).show()
```

## Q9. Explain Predicate Pushdown

Predicate Pushdown is an optimization technique used with Parquet files where filtering conditions are applied while reading data from storage. Only the required rows are loaded into memory, reducing disk I/O, memory usage, and execution time.

In [ ]:
# Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax). 

from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("price") * 1.18
)

df.select("product_id", "price", "final_price").show(5)

+---------------+--------+------------------+
|     product_id|   price|       final_price|
+---------------+--------+------------------+
|FUR-BO-10001798|  261.96|309.11279999999994|
|FUR-CH-10000454|  731.94|          863.6892|
|OFF-LA-10000240|   14.62|           17.2516|
|FUR-TA-10000577|957.5775|        1129.94145|
|OFF-ST-10000760|  22.368|26.394239999999996|
+---------------+--------+------------------+
only showing top 5 rows


## Q11. Difference between Transformations and Actions

### Transformations
Transformations create a new DataFrame but are evaluated lazily.

Examples:
- filter()
- select()

### Actions
Actions trigger the execution of transformations and return results.

Examples:
- show()
- count()

## Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output". 

from pyspark.sql.functions import col

spark.read.parquet(
    "path/to/input"
).filter(
    col("user_id").isNotNull()
).write.mode("overwrite").option(
    "header",
    True
).csv("path/to/output")

## Q13. Difference between Client Mode and Cluster Mode

### Client Mode
- Driver runs on the client machine.
- Suitable for development and testing.

### Cluster Mode
- Driver runs inside the cluster.
- Better fault tolerance.
- Preferred for production environments.

In [ ]:
# Q14: Write a query to filter a dataset for rows where the region is 'North' OR the category is 'Technology'. 

from pyspark.sql.functions import col

df.filter(
    (col("Region") == "North") |
    (col("Category") == "Technology")
).show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+--------+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|         City|         State|Postal Code| Region|     product_id|  category|Sub-Category|        Product Name|   price|Quantity|Discount|  Profit|       final_price|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+----------+------------+--------------------+--------+--------+--------+--------+------------------+
|     8|CA-2014-115812|  6/9/2014| 6/14/2014|Standard Class|   BH-11710|   Brosina Hoffman|   Consumer|United States|  Los Angeles|    California|     

## Q15. Why is `.show(5)` safer than `.collect()` on a multi-terabyte dataset?

The `show(5)` function displays only the first few rows of a DataFrame, making it memory-efficient and suitable for large datasets.

In contrast, `collect()` retrieves the entire dataset to the Driver node. On very large datasets, this can consume excessive memory and may lead to application failure or OutOfMemory errors.

Therefore, `show(5)` is recommended for exploring large datasets.


# Conclusion

In this assignment, Apache Spark architecture and its execution model were studied. Data was loaded, transformed, filtered, and analyzed using Spark DataFrames. Lazy Evaluation, Transformations, Actions, Execution Plans, and optimized storage formats such as CSV and Parquet were explored. The assignment demonstrates practical understanding of distributed data processing using Apache Spark.